# Autoencoder (AE) for icon generation

**Lesson 1: a deterministic autoencoder with just two latent numbers.**

Training: **image → encoder → (z1, z2) → decoder → reconstructed image**.
Generation: **choose (z1, z2) → decoder → image**.

No words, no mu/sigma, no random sampling inside the encoder, and no KL loss. The same input image always has the same encoded coordinates after training.

A regular AE CAN generate by decoding new coordinates. What it does not learn here is a known distribution to sample those coordinates from. Near learned codes, results may be good; arbitrary points can produce poor or mixed icons. Do not assume an AE must always generate worse than a VAE.

Run the cells in order. The lessons use the same 10,000 fixed icons, CNN image paths, and 40 epochs. AE and pure VAE use two-dimensional bottlenecks; the conditioned VAE retains eight dimensions for better detail.


## 1. Download the pre-generated data

This is the same fixed dataset used by the VAE and conditioned VAE lessons.


In [ ]:
# Download the pre-generated training data. No image generator runs here.
from pathlib import Path
import hashlib
import urllib.request

DATA_URL = 'https://raw.githubusercontent.com/dujing82-blip/tiny-black-white-generative-ai/main/data/icons_cnn_v2.npz'
DATA_FILE = Path('icons_cnn_v2.npz')
DATA_SHA256 = 'f5efcac36774ebced04128dc85d297cca9fc11b5469b2baa9be459c00b834006'

def checksum_matches(path):
    return path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() == DATA_SHA256

# Reuse the local file when this cell is run again in the same Colab session.
if not checksum_matches(DATA_FILE):
    temporary = DATA_FILE.with_suffix('.download')
    urllib.request.urlretrieve(DATA_URL, temporary)
    if not checksum_matches(temporary):
        temporary.unlink(missing_ok=True)
        raise ValueError('Dataset checksum mismatch. Please rerun this cell.')
    temporary.replace(DATA_FILE)

print('Training dataset ready:', DATA_FILE, '(10,000 pre-generated icons)')


## 2. Load images only

Labels are stored for the final visualization, but are never given to the AE.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

keras.utils.set_random_seed(42)

with np.load(DATA_FILE, allow_pickle=False) as data:
    X = data['images'].astype('float32') / 255.0
    plot_labels = data['labels'].copy()   # Used ONLY to color the final scatter plot.
    plot_names = data['words'].tolist()

assert X.shape == (10000, 16, 16)
assert np.isin(X, [0.0, 1.0]).all()
X = X[..., np.newaxis]                   # CNN input: (10000, 16, 16, 1).
print('Training images:', X.shape)

# Show a random selection of the training images.
indices = np.random.default_rng(42).choice(len(X), 15, replace=False)
fig, axes = plt.subplots(3, 5, figsize=(8, 5))
for ax, index in zip(axes.flat, indices):
    ax.imshow(X[index, :, :, 0], cmap='gray', vmin=0, vmax=1, interpolation='nearest')
    ax.axis('off')
plt.tight_layout()
plt.show()


## 3. A plain CNN encoder

Compress one image into exactly two numbers. There are no distribution parameters or sampling layer.


In [ ]:
LATENT_DIM = 2

image_input = keras.Input(shape=(16, 16, 1), name='image')
x = layers.Conv2D(32, 3, strides=2, padding='same', activation='relu')(image_input)
x = layers.Conv2D(64, 3, strides=2, padding='same', activation='relu')(x)
x = layers.Flatten()(x)
x = layers.Dense(128, activation='relu')(x)

# Two ordinary deterministic coordinates. Linear output; no enforced range.
z = layers.Dense(LATENT_DIM, name='latent_coordinates')(x)
encoder = keras.Model(image_input, z, name='AE_encoder')
encoder.summary()


## 4. Decoder: two numbers → an icon

This is the same decoder structure used in the pure VAE lesson.


In [ ]:
# ONE input: the two latent coordinates. No word is supplied.
z_input = keras.Input(shape=(2,), name='z')
x = layers.Dense(4 * 4 * 64, activation='relu')(z_input)
x = layers.Reshape((4, 4, 64))(x)
x = layers.Conv2DTranspose(64, 3, strides=2, padding='same', activation='relu')(x)
x = layers.Conv2DTranspose(32, 3, strides=2, padding='same', activation='relu')(x)
output_image = layers.Conv2D(1, 3, padding='same', activation='sigmoid')(x)

decoder = keras.Model(z_input, output_image, name='AE_decoder')
decoder.summary()


## 5. Train with reconstruction loss only

The target is the original image itself. Unlike the VAE, there is no KL term.


In [ ]:
# Connect the encoder and decoder into one ordinary Keras model.
autoencoder = keras.Model(image_input, decoder(encoder(image_input)), name='autoencoder')

def reconstruction_loss(original, reconstructed):
    # Same pixel loss and scale as the VAE lessons: sum the 256 pixel BCEs.
    pixel_loss = keras.losses.binary_crossentropy(original, reconstructed)
    return tf.reduce_sum(pixel_loss, axis=(1, 2))

autoencoder.compile(optimizer=keras.optimizers.Adam(0.001), loss=reconstruction_loss)

# Input = original image; target = the SAME original image.
dataset = tf.data.Dataset.from_tensor_slices((X, X))
dataset = dataset.shuffle(len(X), seed=42).batch(128).prefetch(tf.data.AUTOTUNE)
EPOCHS = 40
history = autoencoder.fit(dataset, epochs=EPOCHS)


## 6. First check reconstruction

The decoder has practiced decoding the coordinates produced by the encoder. Reconstruction tests that learned behavior.


In [ ]:
# Show one example from each known class, ONLY for evaluation.
indices = [np.where(plot_labels == label)[0][0] for label in range(5)]
examples = X[indices]
codes = encoder(examples, training=False).numpy()
reconstructed = decoder(codes, training=False).numpy()
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i in range(5):
    axes[0, i].imshow(examples[i, :, :, 0], cmap='gray', vmin=0, vmax=1)
    axes[0, i].set_title(plot_names[i])
    axes[1, i].imshow(reconstructed[i, :, :, 0], cmap='gray', vmin=0, vmax=1)
    axes[1, i].set_title(f'z=({codes[i,0]:.1f}, {codes[i,1]:.1f})', fontsize=9)
    axes[0, i].axis('off')
    axes[1, i].axis('off')
plt.suptitle('Top: original. Bottom: reconstruction from its encoded coordinates.')
plt.tight_layout()
plt.show()


## 7. Type two numbers and generate

Now the encoder is not used. Try arbitrary values, then try the coordinates printed above. A new coordinate pair still produces an output, but it may be outside the region the AE learned.


In [ ]:
def generate(z1, z2, binary=True):
    point = np.array([[z1, z2]], dtype='float32')
    if not np.isfinite(point).all():
        raise ValueError('Please enter two finite numbers.')
    # Generation uses ONLY the decoder, with no input image and no word.
    probabilities = decoder(point, training=False).numpy()[0, :, :, 0]
    image = probabilities > 0.5 if binary else probabilities
    plt.figure(figsize=(3, 3))
    plt.imshow(image, cmap='gray', vmin=0, vmax=1, interpolation='nearest')
    plt.title(f'z1 = {z1:.2f}, z2 = {z2:.2f}')
    plt.axis('off')
    plt.show()
    return probabilities

# Change these two values and run again.
_ = generate(z1=0.5, z2=-1.0)


## 8. Interactive generator

The Random z button draws from N(0, I), just as in the VAE lesson. **This is an experiment for the AE, not a distribution it was trained to match.**


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

z1_box = widgets.FloatText(value=0.5, description='z1:')
z2_box = widgets.FloatText(value=-1.0, description='z2:')
binary_box = widgets.Checkbox(value=True, description='Black/white display')
generate_button = widgets.Button(description='Generate icon')
random_button = widgets.Button(description='Random z')
output = widgets.Output()

def draw(_=None):
    with output:
        clear_output(wait=True)
        try:
            _ = generate(z1_box.value, z2_box.value, binary=binary_box.value)
        except ValueError as error:
            print(error)

def randomize(_):
    z1_box.value, z2_box.value = np.random.normal(size=2)
    draw()

generate_button.on_click(draw)
random_button.on_click(randomize)
display(widgets.VBox([widgets.HBox([z1_box, z2_box]), binary_box,
                      widgets.HBox([generate_button, random_button]), output]))
draw()


## 9. Where are the learned codes?

The AE has no reason to center its codes at zero or keep their scale near one. Labels only color this plot after training.


In [ ]:
encoded = encoder.predict(X, batch_size=256, verbose=0)
plt.figure(figsize=(7, 6))
for label, name in enumerate(plot_names):
    points = encoded[plot_labels == label]
    plt.scatter(points[:, 0], points[:, 1], s=5, alpha=0.3, label=name)
plt.xlabel('z1')
plt.ylabel('z2')
plt.title('AE: deterministic codes of training images')
plt.legend()
plt.grid(alpha=0.2)
plt.show()


## 10. Explore the learned coordinate range

Each tile is a new decoded point. We use the 1st–99th percentiles of the actual codes for the axes. Even inside this rectangle, gaps between learned codes can produce poor shapes.


In [ ]:
steps = 15
lower, upper = np.percentile(encoded, [1, 99], axis=0)
x_values = np.linspace(lower[0], upper[0], steps)
y_values = np.linspace(upper[1], lower[1], steps)
points = np.array([[a, b] for b in y_values for a in x_values], dtype='float32')
images = decoder(points, training=False).numpy()[:, :, :, 0]
canvas = (images > 0.5).reshape(steps, steps, 16, 16).transpose(0, 2, 1, 3).reshape(steps*16, steps*16)
dx = (x_values[-1] - x_values[0]) / (2 * (steps - 1))
dy = (y_values[0] - y_values[-1]) / (2 * (steps - 1))
plt.figure(figsize=(9, 9))
plt.imshow(canvas, cmap='gray', vmin=0, vmax=1, interpolation='nearest',
           extent=[lower[0]-dx, upper[0]+dx, lower[1]-dy, upper[1]+dy], aspect='auto')
plt.xlabel('z1')
plt.ylabel('z2')
plt.title('AE generation across the observed coordinate range')
plt.show()


## 11. Compare with sampling near zero

Unlike a VAE, this AE does not regularize its codes toward a standard normal distribution. Compare this result with reconstruction and with the actual code locations.


In [ ]:
def show_latent_grid(limit=3.0, steps=15, binary=True):
    horizontal = np.linspace(-limit, limit, steps)
    vertical = np.linspace(limit, -limit, steps)  # Larger z2 at the top.
    points = np.array([[a, b] for b in vertical for a in horizontal], dtype='float32')
    images = decoder(points, training=False).numpy()[:, :, :, 0]
    if binary:
        images = images > 0.5
    # Arrange the generated images into a single tiled image.
    canvas = images.reshape(steps, steps, 16, 16).transpose(0, 2, 1, 3).reshape(steps*16, steps*16)
    # Tile centers correspond exactly to the sampled coordinates.
    half_step = (horizontal[1] - horizontal[0]) / 2
    edge = limit + half_step
    plt.figure(figsize=(9, 9))
    plt.imshow(canvas, cmap='gray', vmin=0, vmax=1, interpolation='nearest',
               extent=[-edge, edge, -edge, edge])
    plt.xlabel('z1')
    plt.ylabel('z2')
    plt.title('AE: coordinates from −3 to +3 (no learned normal prior)')
    plt.show()

show_latent_grid()


## Questions for students

1. Does good reconstruction guarantee good generation at arbitrary coordinates?
2. What happens between two known codes? Try their midpoint with `generate`.
3. Why might coordinates near zero be a poor place to sample this AE?
4. What changes when the next lesson adds sampling and KL regularization?

**An AE decoder can generate an image from a new code. A VAE additionally learns with a distribution and a prior, making random sampling more principled. Neither guarantees that every coordinate will produce a recognizable icon.**
